# Queues

**Reference**
- `deque` - what it is and when to use it
- `deque` operations and time complexity
- Monotonic deque
- Priority queue (`heapq`)

**Problems**
- LC 232 - Implement Queue using Stacks
- LC 933 - Number of Recent Calls
- LC 239 - Sliding Window Maximum

## Reference

### `deque` is the queue

`from collections import deque`

A doubly ended queue - append and pop at *either* end in O(1). FIFO means push on the
right, pop from the left:

```python
q = deque()       # deque([1, 2, 3]) to seed it
q.append(x)       # enqueue      O(1)
q.popleft()       # dequeue      O(1)
q[0]              # peek front   O(1)
if not q:         # empty?
```

A list looks like it works, but `list.pop(0)` shifts every remaining element - O(n) per
dequeue, O(n^2) overall. Use a list for a stack, a `deque` for a queue.

### `deque` operations and time complexity

| Operation | Example | Time |
|---|---|---|
| Push either end | `q.append(x)`, `q.appendleft(x)` | O(1) |
| Pop either end | `q.pop()`, `q.popleft()` | O(1) |
| Peek either end | `q[0]`, `q[-1]` | O(1) |
| Index in the middle | `q[i]` | O(n) |
| Membership | `x in q` | O(n) |
| Size | `len(q)` | O(1) |
| Rotate | `q.rotate(k)` | O(k) |
| Build from iterable | `deque(arr)` | O(n) |

`deque(maxlen=n)` is bounded: pushing onto a full deque drops the element at the far end,
which gives you a fixed-size window for free.

### Monotonic deque

The queue version of the monotonic stack, and the way to get a sliding window's
max (or min) in O(1). Keep the deque decreasing by popping from the **right** anything
smaller than the incoming value, and popping from the **left** whatever has slid out of
the window. Store *indices*, not values, so you can tell when something leaves.

```python
while dq and arr[dq[-1]] < x:   # x is bigger and newer -> those can never be the max
    dq.pop()
dq.append(i)

if dq[0] <= i - k:              # front has fallen out of the window
    dq.popleft()

arr[dq[0]]                      # window max
```

Each index is pushed and popped once -> O(n). Decreasing deque for the max, increasing
for the min.

### Priority queue (`heapq`)

Not FIFO - pops the *smallest* item, not the oldest. It's a plain list kept in heap
order by the functions:

```python
import heapq

h = []
heapq.heappush(h, x)     # O(log n)
heapq.heappop(h)         # smallest   O(log n)
h[0]                     # peek min   O(1)
heapq.heapify(arr)       # in place   O(n)
```

Only a min-heap exists - push `-x` (or `(-key, item)`) for a max-heap.

BFS is the other place a queue always shows up - see `7 - Trees`.

## Implement Queue using Stacks (LC 232)

In [ ]:
class MyQueue:
    def __init__(self):
        self.inS = []   # pushes land here, newest on top
        self.outS = []  # reversed, so the oldest is on top

    def push(self, x):
        self.inS.append(x)

    def pop(self):
        self._shift()
        return self.outS.pop()

    def peek(self):
        self._shift()
        return self.outS[-1]

    def empty(self):
        return not self.inS and not self.outS

    def _shift(self):
        if not self.outS:          # only when out is empty, or the order breaks
            while self.inS:
                self.outS.append(self.inS.pop())

Every element is moved across at most once, so `pop`/`peek` are O(1) amortised even
though a single call can cost O(n).

## Number of Recent Calls (LC 933)

In [ ]:
from collections import deque

class RecentCounter:
    def __init__(self):
        self.q = deque()   # ping times, oldest on the left

    def ping(self, t):
        self.q.append(t)

        while self.q[0] < t - 3000:   # older than the window -> gone for good
            self.q.popleft()

        return len(self.q)

## Sliding Window Maximum (LC 239)

In [ ]:
from collections import deque

def maxSlidingWindow(nums, k):
    res, dq = [], deque()   # decreasing deque of indices

    for i, n in enumerate(nums):
        while dq and nums[dq[-1]] < n:
            dq.pop()
        dq.append(i)

        if dq[0] <= i - k:          # front slid out of the window
            dq.popleft()
        if i >= k - 1:              # first full window reached
            res.append(nums[dq[0]])

    return res